# Notebook 7 — Presets & Recipes

Presets are the shorthand layer of Blueprint. Every preset is a plain Python function that returns a fully configured `Class` or `Influence` — nothing more. You can always replace a preset with its raw equivalent and the result will be identical.

This notebook covers:
- The philosophy behind presets: convenience, not magic
- **Class presets**: `RandomClass`, `HighValueClass`, `LowValueClass`, `OutlierClass`
- **Influence presets**: `ScalesWith`, `CorrelatedWith`, `Caps`
- Combining presets with custom-built components
- When to reach for a preset vs. the raw API

In [1]:
from blueprint import Blueprint, Feature, Class, Influence
from blueprint.presets import (
    RandomClass, HighValueClass, LowValueClass, OutlierClass,
    ScalesWith, CorrelatedWith, Caps,
)

import numpy as np
import pandas as pd

## Philosophy: Presets Are Just Python Wrappers

Every preset is a function that calls the same public API you would call yourself. There is no hidden logic, no special runtime behavior, no registry. `RandomClass("vip", p=0.1)` is exactly the same as:

```python
Class("vip", when=("__random__", 0.1))
```

Presets exist for one reason: to name common patterns. Instead of remembering that `when=("__random__", p)` is how you select a random fraction of rows, you write `RandomClass(name, p=...)` and the intent is immediately legible.

You can always peek at what a preset returns:

In [2]:
rc = RandomClass("vip", p=0.15)
print(type(rc))   # it's just a Class
print(rc.when)    # the condition it wraps

<class 'blueprint.core.klass.Class'>
('__random__', 0.15)


Because presets return standard `Class` or `Influence` objects, you can chain `.override()` onto a class preset, or pass an influence preset to `add_influence()`, exactly as you would with a hand-built component.

## Class Presets

Class presets cover the four most common ways to carve a population into segments:

| Preset | Selects rows by |
|---|---|
| `RandomClass(name, p)` | Random probability `p` |
| `HighValueClass(name, feature, top_pct)` | Top `top_pct` fraction of a column |
| `LowValueClass(name, feature, bottom_pct)` | Bottom `bottom_pct` fraction of a column |
| `OutlierClass(name, p, features, magnitude)` | Random probability `p` (marks outlier intent) |

### `RandomClass`

`RandomClass(name, p)` selects each row independently at probability `p`. It is a direct shorthand for `Class(name, when=("__random__", p))`.

Use it whenever a segment has no deterministic column criterion — churn risk, promotional eligibility, test/control assignment.

In [3]:
bp = (
    Blueprint(n=10, seed=42)
    .add_feature(
        Feature("product", dtype="category", values=["A", "B", "C"]),
        Feature("price",   dtype=float, base=100, std=20, clip=(0, None)),
    )
    .add_class(
        RandomClass("premium", p=0.3).override("price", base=200, std=15)
    )
)

df = bp.emit()
print(df.to_string())
print(f"\npremium rows (price > 150): {(df['price'] > 150).sum()} out of {len(df)}")

  product       price
0       A   73.956410
1       C  201.416496
2       B   93.675148
3       B   99.663977
4       B   82.939121
5       C  117.587959
6       A  115.555839
7       C  225.969478
8       A  195.728496
9       A  109.350187

premium rows (price > 150): 3 out of 10


About 30 % of rows received the premium price distribution. The exact rows are determined by the master `seed`, so re-running with the same seed produces identical assignments.

**Raw equivalent** for reference:
```python
Class("premium", when=("__random__", 0.3)).override("price", base=200, std=15)
```

### `HighValueClass`

`HighValueClass(name, feature, top_pct)` selects the top `top_pct` fraction of rows by the value of `feature`. Under the hood it uses a callable `when=` that computes the quantile at `1.0 - top_pct`.

Use it to model VIP customers, high-revenue accounts, top-performing products — anything where "high value" is defined by a column threshold.

In [4]:
bp = (
    Blueprint(n=12, seed=42)
    .add_feature(
        Feature("revenue", dtype=float, base=50000, std=20000, clip=(0, None)),
        Feature("tier",    dtype="category", values=["standard"], weights=[1.0]),
    )
    .add_class(
        HighValueClass("whale", feature="revenue", top_pct=0.25)
        .override("tier", values=["vip"], weights=[1.0])
    )
)

df = bp.emit()
threshold = df["revenue"].quantile(0.75)
print(df.to_string())
print(f"\nVIP rows: {(df['tier'] == 'vip').sum()}  (top 25% threshold: {threshold:,.0f})")

         revenue      tier
0   56094.341595  standard
1   29200.317875  standard
2   65009.023916  standard
3   68811.294328       vip
4   10979.296227  standard
5   23956.409863  standard
6   52556.808063  standard
7   43675.148153  standard
8   49663.976850  standard
9   32939.121449  standard
10  67587.959497       vip
11  65555.838709       vip

VIP rows: 3  (top 25% threshold: 65,146)


The three highest-revenue rows received `tier='vip'`. Because the quantile is computed from the actual generated data, the exact cut-off adapts to the data rather than being fixed at a hardcoded value.

**Raw equivalent**:
```python
Class("whale", when=lambda df: df["revenue"] >= df["revenue"].quantile(0.75))
    .override("tier", values=["vip"], weights=[1.0])
```

### `LowValueClass`

`LowValueClass(name, feature, bottom_pct)` mirrors `HighValueClass` in the opposite direction: it selects the bottom `bottom_pct` fraction of rows by a column’s value.

Use it for at-risk customers, low-performing agents, budget-tier items.

In [5]:
bp = (
    Blueprint(n=10, seed=42)
    .add_feature(
        Feature("score",     dtype=float, base=70, std=15, clip=(0, 100)),
        Feature("risk_flag", dtype=bool, p=0.05),
    )
    .add_class(
        LowValueClass("high_risk", feature="score", bottom_pct=0.2)
        .override("risk_flag", p=1.0)
    )
)

df = bp.emit()
flagged_scores = df.loc[df["risk_flag"], "score"].round(2).tolist()
print(df.to_string())
print(f"\nFlagged rows (bottom 20%): {df['risk_flag'].sum()}")
print(f"Their scores: {flagged_scores}")

       score  risk_flag
0  74.570756      False
1  54.400238      False
2  81.256768      False
3  84.108471      False
4  40.734472       True
5  50.467307       True
6  71.917606      False
7  65.256361      False
8  69.747983      False
9  57.204341      False

Flagged rows (bottom 20%): 2
Their scores: [40.73, 50.47]


The two rows with the lowest `score` values were guaranteed a `risk_flag=True`. All other rows used the default `p=0.05` probability.

**Raw equivalent**:
```python
Class("high_risk", when=lambda df: df["score"] <= df["score"].quantile(0.2))
    .override("risk_flag", p=1.0)
```

### `OutlierClass`

`OutlierClass(name, p, features, magnitude)` returns a random class (same mechanism as `RandomClass`) and attaches `_outlier_features` and `_outlier_magnitude` as metadata attributes on the `Class` object. Blueprint’s core does not read those attributes automatically — they are documentation for the intent, not a wired-in behavior.

To actually produce outlier values you pair `OutlierClass` with `.override()` calls that push the flagged rows to extreme distributions:

In [6]:
bp = (
    Blueprint(n=20, seed=7)
    .add_feature(
        Feature("age",    dtype=int,   base=40, std=10, clip=(18, 80)),
        Feature("income", dtype=float, base=60000, std=15000, clip=(0, None)),
    )
    .add_class(
        OutlierClass("spike", p=0.15, features=["income"], magnitude=3.0)
        .override("income", base=250000, std=20000)
    )
)

df = bp.emit()
print(df.to_string())
print(f"\nOutlier rows (income > 150,000): {(df['income'] > 150_000).sum()}")

    age         income
0    40   32373.974433
1    43   56473.633034
2    37   40988.302778
3    31   64068.965382
4    35  255636.704180
5    30   57196.035831
6    41   22248.604338
7    53   51919.606562
8    35   59272.485819
9    34   61699.634790
10   45   37047.963517
11   44   52833.700859
12   41  244007.439692
13   31   47867.441409
14   40   75913.479351
15   47  228807.130655
16   27  253456.979155
17   35   73265.848011
18   21   51245.993509
19   27  208184.809723

Outlier rows (income > 150,000): 5


The `_outlier_features` and `_outlier_magnitude` parameters are metadata that travel with the object — you could read them in downstream tooling to understand the blueprint’s intent, but the data-generation behavior comes entirely from the `.override()` call.

**Raw equivalent**:
```python
Class("spike", when=("__random__", 0.15)).override("income", base=250000, std=20000)
```

## Influence Presets

Influence presets cover three common causal patterns:

| Preset | Effect |
|---|---|
| `ScalesWith(source, target, rate)` | `target += rate * source` (linear scaling) |
| `CorrelatedWith(source, target, correlation)` | Nudges `target` to track `source` with the given Pearson correlation |
| `Caps(source, target, threshold, decay)` | Applies a diminishing-returns cap on `target` when `source` exceeds `threshold` |

### `ScalesWith`

`ScalesWith(source, target, rate)` is the simplest influence preset: it adds `rate * source_value` to each row’s target value. This is the `"+{rate} per unit"` effect string written as a function call.

Use it for deterministic linear relationships: price per square foot, revenue per unit sold, commission rate.

In [7]:
bp = (
    Blueprint(n=8, seed=42)
    .add_feature(
        Feature("sqft",  dtype=int,   base=1800, std=400, clip=(500, 5000)),
        Feature("price", dtype=float, base=0,    std=0,   derived=True),
    )
    .add_influence(ScalesWith("sqft", "price", rate=175))
)

df = bp.emit()
print(df.to_string())
print(f"\nprice / sqft ratios: {(df['price'] / df['sqft']).round(1).tolist()}")

   sqft     price
0  1922  336350.0
1  1384  242200.0
2  2100  367500.0
3  2176  380800.0
4  1020  178500.0
5  1279  223825.0
6  1851  323925.0
7  1674  292950.0

price / sqft ratios: [175.0, 175.0, 175.0, 175.0, 175.0, 175.0, 175.0, 175.0]


Every row has exactly `price = sqft * 175` — a clean, deterministic linear relationship.

**Raw equivalent**:
```python
Influence("sqft").on("price", effect="+175 per unit")
```

### `CorrelatedWith`

`CorrelatedWith(source, target, correlation)` nudges `target` so that it tracks `source` with approximately the given Pearson correlation coefficient. It normalizes `source`, scales by `target`’s own standard deviation, and adds the result to `target`.

The correlation is approximate — with small datasets it will vary, and downstream transformations such as clipping can reduce the realized Pearson correlation from the requested value.

Use it when you want two columns to move together without locking them into a rigid linear formula.

In [8]:
bp = (
    Blueprint(n=10, seed=42)
    .add_feature(
        Feature("ad_spend", dtype=float, base=10000, std=4000, clip=(0, None)),
        Feature("revenue",  dtype=float, base=80000, std=25000, clip=(0, None)),
    )
    .add_influence(CorrelatedWith("ad_spend", "revenue", correlation=0.75))
)

df = bp.emit()
r = np.corrcoef(df["ad_spend"], df["revenue"])[0, 1]
print(df.to_string())
print(f"\ncorr(ad_spend, revenue) = {r:.3f}")

       ad_spend        revenue
0  11218.868319  111291.054095
1   5840.063575   89206.694140
2  13001.804783   97435.278518
3  13762.258866  126728.696676
4   2195.859245   68208.194766
5   4791.281973   44468.773458
6  10511.361613   95954.106362
7   8735.029631   56308.863166
8   9932.795370  106594.339222
9   6587.824290   71230.781291

corr(ad_spend, revenue) = 0.757


In [9]:
# At larger n the realized correlation converges on the requested value
bp_large = (
    Blueprint(n=2000, seed=42)
    .add_feature(
        Feature("ad_spend", dtype=float, base=10000, std=4000, clip=(0, None)),
        Feature("revenue",  dtype=float, base=80000, std=25000, clip=(0, None)),
    )
    .add_influence(CorrelatedWith("ad_spend", "revenue", correlation=0.75))
)
df_large = bp_large.emit()
r_large = np.corrcoef(df_large["ad_spend"], df_large["revenue"])[0, 1]
print(f"n=2000: corr = {r_large:.3f}")

n=2000: corr = 0.603


**Raw equivalent** — shown here to illustrate what `CorrelatedWith` actually does:
```python
def _corr_fn(source_col, target_col, df):
    src_std = source_col.std()
    if src_std < 1e-10:
        return target_col
    src_norm = (source_col - source_col.mean()) / src_std
    tgt_std = target_col.std()
    return target_col + 0.75 * src_norm * tgt_std

Influence("ad_spend").on("revenue", fn=_corr_fn)
```

### `Caps`

`Caps(source, target, threshold, decay)` applies a diminishing-returns cap: once `source` exceeds `threshold`, the `target` value is multiplied by a factor that shrinks as the excess grows.

The factor is: `1 / (1 + decay * max(source - threshold, 0))`

- Below `threshold`, the cap factor is exactly 1.0 — `target` is untouched.
- Above `threshold`, the factor falls toward zero as the excess grows.
- `decay` controls how steeply the cap takes effect.

Use it to model saturation effects: training time vs. model accuracy, marketing spend vs. incremental lift, years of experience vs. salary.

In [10]:
bp = (
    Blueprint(n=10, seed=42)
    .add_feature(
        Feature("years_exp", dtype=float, base=8, std=5, clip=(0, 25)),
        Feature("salary",    dtype=float, base=80000, std=0),
    )
    .add_influence(Caps("years_exp", "salary", threshold=10, decay=0.05))
)

df = bp.emit()
print(df[["years_exp", "salary"]].sort_values("years_exp").round(0).to_string())

# Show the decay formula directly
for exp in [12, 13, 20]:
    factor = 1.0 / (1.0 + 0.05 * max(exp - 10, 0))
    print(f"cap factor at {exp} years: {factor:.3f}")

   years_exp   salary
4        0.0  80000.0
5        1.0  80000.0
1        3.0  80000.0
9        4.0  80000.0
7        6.0  80000.0
8        8.0  80000.0
6        9.0  80000.0
0       10.0  80000.0
2       12.0  73556.0
3       13.0  70476.0
cap factor at 12 years: 0.909
cap factor at 13 years: 0.870
cap factor at 20 years: 0.667


Rows with `years_exp <= 10` retain the full base salary of 80,000. Above the threshold the salary is reduced by the cap factor — 12 years gets ~91% of base (factor ≈ 0.909), 20 years gets ~67% (factor ≈ 0.667).

**Raw equivalent**:
```python
def _cap_fn(source_col, target_col, df):
    excess = np.maximum(source_col - 10, 0)
    factor = 1.0 / (1.0 + 0.05 * excess)
    return target_col * factor

Influence("years_exp").on("salary", fn=_cap_fn)
```

## Combining Presets with Custom Components

Presets and raw components are fully interchangeable. A realistic blueprint will often mix both: presets for common patterns, raw `Class` or `Influence` objects for anything domain-specific.

Here is a workforce dataset that combines four preset components with one custom influence:

In [11]:
bp = (
    Blueprint(n=500, seed=42)
    .add_feature(
        Feature("dept",        dtype="category",
                values=["Engineering", "Sales", "HR", "Finance"],
                weights=[0.4, 0.3, 0.15, 0.15]),
        Feature("years_exp",   dtype=float, base=7, std=4, clip=(0, 30)),
        Feature("performance", dtype=float, base=3.0, std=0.7, clip=(1.0, 5.0)),
        Feature("salary",      dtype=float, base=90000, std=20000, clip=(30000, None)),
    )
    # Preset class: senior employees in the top 20% by experience
    .add_class(
        HighValueClass("senior", feature="years_exp", top_pct=0.2)
        .override("salary", base=130000, std=15000)
    )
    # Preset class: ~10% of employees are top performers
    .add_class(
        RandomClass("top_performer", p=0.1)
        .override("performance", base=4.8, std=0.1, clip=(4.5, 5.0))
    )
    # Preset influence: each performance point adds 5,000 to salary
    .add_influence(ScalesWith("performance", "salary", rate=5000))
    # Preset influence: salary growth flattens after 20 years of experience
    .add_influence(Caps("years_exp", "salary", threshold=20, decay=0.02))
)

df = bp.emit()
print(df.head(10).round(2).to_string())
print(f"\nShape: {df.shape}")
print(f"Top performers (perf >= 4.5):  {(df['performance'] >= 4.5).sum()}")
print(f"Senior employees (top 20% exp): {(df['years_exp'] >= df['years_exp'].quantile(0.8)).sum()}")
print(f"Salary range: {df['salary'].min():,.0f} \u2014 {df['salary'].max():,.0f}")

          dept  years_exp  performance     salary
0           HR      12.73         1.93  152144.18
1        Sales       7.37         3.16  108094.87
2      Finance       9.32         3.51  114173.41
3        Sales       6.77         3.26  121862.54
4  Engineering       6.32         3.44   91022.21
5      Finance       3.88         2.02  117646.23
6           HR       8.72         3.23   78698.58
7           HR       3.59         4.89  121018.08
8  Engineering       9.66         4.77  117400.93
9        Sales      11.34         3.61  147980.90

Shape: (500, 4)
Top performers (perf >= 4.5):  58
Senior employees (top 20% exp): 100
Salary range: 51,070 — 183,253


All four presets plug into the same `add_class` / `add_influence` calls as hand-built components. The DAG evaluation order handles the interaction between `ScalesWith` (performance → salary) and `Caps` (experience → salary) automatically.

## When to Use Presets vs. Raw API

Presets are a naming layer, not a capability layer. Anything a preset does, the raw API can do too. The question is readability.

**Reach for a preset when:**
- The intent maps exactly to the preset name (`HighValueClass`, `ScalesWith`)
- You’re building a blueprint quickly and want self-documenting code
- The pattern is common enough that a reader will recognize it immediately

**Reach for the raw API when:**
- The preset’s parameters don’t quite fit your use case
- You need a non-standard condition type (e.g., `when=("col", "between", (lo, hi))`)
- You want a fully custom `fn=` influence that doesn’t fit `ScalesWith`, `CorrelatedWith`, or `Caps`
- You want the code to be fully explicit with no indirection

Neither form is more correct. Presets are shorter; raw API is more transparent. Mix them freely.

---

## Summary

**Class presets**

| Preset | Raw equivalent |
|---|---|
| `RandomClass(name, p)` | `Class(name, when=("__random__", p))` |
| `HighValueClass(name, feature, top_pct)` | `Class(name, when=lambda df: df[feature] >= df[feature].quantile(1 - top_pct))` |
| `LowValueClass(name, feature, bottom_pct)` | `Class(name, when=lambda df: df[feature] <= df[feature].quantile(bottom_pct))` |
| `OutlierClass(name, p, features, magnitude)` | `Class(name, when=("__random__", p))` + `.override(...)` |

**Influence presets**

| Preset | Raw equivalent |
|---|---|
| `ScalesWith(source, target, rate)` | `Influence(source).on(target, effect=f"+{rate} per unit")` |
| `CorrelatedWith(source, target, correlation)` | `Influence(source).on(target, fn=<normalize-and-nudge fn>)` |
| `Caps(source, target, threshold, decay)` | `Influence(source).on(target, fn=<diminishing-returns fn>)` |

All presets return standard `Class` or `Influence` objects. `.override()` and `.on()` chaining work on preset results exactly as they do on raw objects.